# Train weather-transfer and operational energy models

Two deliberately different models are trained from the CSV feature candidates:

1. **Transfer/weather model:** calendar + current/forecast weather + past weather features; **no load lags**. It is tested by holding out an entire city.
2. **Operational model:** the same transfer feature set plus `load_lag_1h`, `load_lag_24h`, and `load_lag_168h`. It is trained and evaluated forward in time within each city, where those historical loads are available.

All preprocessing is fitted only to each training partition. The model is scikit-learn's `HistGradientBoostingRegressor`, so it runs without LightGBM/CatBoost installation.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

BASE = Path.cwd().parent
DATA_DIR = BASE / 'processed_feature_candidates_cross_city_v1'
RESULTS_DIR = BASE / 'training_results_cross_city_v1'
RESULTS_DIR.mkdir(exist_ok=True)
TARGET = 'hourly_load_mw'
RANDOM_STATE = 42
TIME_TRAIN_FRACTION = 0.80
print('Data:', DATA_DIR)
print('Results:', RESULTS_DIR)

## Exact feature definitions

These are intentionally compact starting sets, not every generated candidate. The candidate files and manifest remain available for later ablation. `TRANSFER_FEATURES` contains no target-derived variables and no city identifier. Current weather values must be replaced with weather forecasts when the deployment horizon is day-ahead.

In [ ]:
CALENDAR_FEATURES = [
    'sin_hour', 'cos_hour', 'sin_dow', 'cos_dow', 'sin_month', 'cos_month', 'sin_doy', 'cos_doy',
    'is_weekend', 'is_holiday_us_federal', 'is_business_hour', 'is_night',
]
CURRENT_WEATHER_FEATURES = [
    'temperature_c', 'dew_point_c', 'station_pressure_kpa', 'wind_speed_ms',
    'wind_direction_sin', 'wind_direction_cos',
    'extraterrestrial_direct_normal_radiation_wh_m2',
    'broadband_aerosol_optical_depth',
]
CROSS_VARIABLE_FEATURES = [
    'temperature_dewpoint_spread_c', 'humidex_c',
    'cooling_degree_base_18c', 'heating_degree_base_18c',
    'cooling_degree_base_22c', 'heating_degree_base_22c',
    'temperature_x_sin_hour', 'temperature_x_cos_hour',
    'humidex_x_sin_hour', 'humidex_x_cos_hour',
]
WEATHER_HISTORY_FEATURES = [
    'temperature_c_lag_1h', 'temperature_c_lag_3h', 'temperature_c_lag_6h',
    'temperature_c_lag_12h', 'temperature_c_lag_24h',
    'temperature_c_prior_6h_mean', 'temperature_c_prior_24h_mean',
    'temperature_c_prior_6h_max', 'temperature_c_prior_24h_max',
    'temperature_change_1h', 'temperature_change_6h', 'temperature_change_24h',
    'dew_point_c_lag_3h', 'dew_point_c_lag_6h', 'dew_point_c_lag_24h',
    'dew_point_c_prior_6h_mean', 'dew_point_c_prior_24h_mean',
    'wind_speed_ms_lag_3h', 'wind_speed_ms_lag_24h',
    'wind_speed_ms_prior_6h_mean', 'wind_speed_ms_prior_24h_mean',
]
TRANSFER_FEATURES = CALENDAR_FEATURES + CURRENT_WEATHER_FEATURES + CROSS_VARIABLE_FEATURES + WEATHER_HISTORY_FEATURES
OPERATIONAL_LOAD_FEATURES = ['load_lag_1h', 'load_lag_24h', 'load_lag_168h']
OPERATIONAL_FEATURES = TRANSFER_FEATURES + OPERATIONAL_LOAD_FEATURES
print(f'Transfer model: {len(TRANSFER_FEATURES)} features')
print(f'Operational model: {len(OPERATIONAL_FEATURES)} features')

In [ ]:
def load_city(city):
    path = DATA_DIR / f'{city}_feature_candidates.csv'
    df = pd.read_csv(path, parse_dates=['timestamp']).sort_values('timestamp').reset_index(drop=True)
    return df

cities = {city: load_city(city) for city in ('las_vegas', 'phoenix')}
for city, df in cities.items():
    missing = sorted(set(OPERATIONAL_FEATURES) - set(df.columns))
    assert not missing, f'{city} missing: {missing}'
    print(city, df.shape, df['timestamp'].min(), 'to', df['timestamp'].max())

## Leakage-safe fitting helpers

The imputer is fit on the training rows only. Weather-lag missingness is represented through missing-value indicators. For the operational model, initial rows without all three required load lags are excluded rather than imputed.

In [ ]:
def build_model():
    preprocess = SimpleImputer(strategy='median', add_indicator=True)
    regressor = HistGradientBoostingRegressor(
        learning_rate=0.06, max_iter=350, max_leaf_nodes=31,
        l2_regularization=1.0, random_state=RANDOM_STATE
    )
    return Pipeline([('imputer', preprocess), ('model', regressor)])

def metrics(y_true, y_pred):
    return {
        'mae_mw': mean_absolute_error(y_true, y_pred),
        'rmse_mw': np.sqrt(mean_squared_error(y_true, y_pred)),
        'r2': r2_score(y_true, y_pred),
        'nmae_percent_of_mean_load': 100 * mean_absolute_error(y_true, y_pred) / np.mean(y_true),
    }

def fit_predict(train_df, test_df, features, operational=False):
    train = train_df.dropna(subset=[TARGET]).copy()
    test = test_df.dropna(subset=[TARGET]).copy()
    if operational:
        train = train.dropna(subset=OPERATIONAL_LOAD_FEATURES)
        test = test.dropna(subset=OPERATIONAL_LOAD_FEATURES)
    model = build_model()
    model.fit(train[features], train[TARGET])
    prediction = model.predict(test[features])
    return model, test, prediction

## 1. Cross-city evaluation: weather-transfer model

No historical-load feature appears in either the inputs or model. This is the main test of transferable weather response.

In [ ]:
cross_city_rows = []
cross_city_predictions = []
for train_city, test_city in [('las_vegas', 'phoenix'), ('phoenix', 'las_vegas')]:
    model, test, pred = fit_predict(cities[train_city], cities[test_city], TRANSFER_FEATURES, operational=False)
    result = {'experiment': 'cross_city_transfer', 'train_city': train_city, 'test_city': test_city, **metrics(test[TARGET], pred)}
    cross_city_rows.append(result)
    prediction_frame = test[['timestamp', 'city', TARGET]].copy()
    prediction_frame['prediction_mw'] = pred
    prediction_frame['train_city'] = train_city
    prediction_frame['experiment'] = 'cross_city_transfer'
    cross_city_predictions.append(prediction_frame)
    joblib.dump(model, RESULTS_DIR / f'transfer_{train_city}_to_{test_city}.joblib')

cross_city_results = pd.DataFrame(cross_city_rows)
display(cross_city_results)

## 2. Forward-in-time operational evaluation

This model is trained separately for each city because it is intended for a city with observed load history. The last 20% of timestamps are held out; no random shuffling is used.

In [ ]:
operational_rows = []
operational_predictions = []
for city, df in cities.items():
    cutoff = df['timestamp'].iloc[int(len(df) * TIME_TRAIN_FRACTION)]
    train_df = df[df['timestamp'] < cutoff]
    test_df = df[df['timestamp'] >= cutoff]
    model, test, pred = fit_predict(train_df, test_df, OPERATIONAL_FEATURES, operational=True)
    operational_rows.append({'experiment': 'within_city_operational', 'train_city': city, 'test_city': city, 'cutoff': cutoff, **metrics(test[TARGET], pred)})
    prediction_frame = test[['timestamp', 'city', TARGET]].copy()
    prediction_frame['prediction_mw'] = pred
    prediction_frame['train_city'] = city
    prediction_frame['experiment'] = 'within_city_operational'
    operational_predictions.append(prediction_frame)
    joblib.dump(model, RESULTS_DIR / f'operational_{city}.joblib')

operational_results = pd.DataFrame(operational_rows)
display(operational_results)

## Save results and inspect feature importance

Permutation importance is calculated only for a sample of the held-out data, after fitting. It is a diagnostic, not the feature-selection procedure itself.

In [ ]:
all_results = pd.concat([cross_city_results, operational_results], ignore_index=True)
all_predictions = pd.concat(cross_city_predictions + operational_predictions, ignore_index=True)
all_results.to_csv(RESULTS_DIR / 'model_metrics.csv', index=False)
all_predictions.to_csv(RESULTS_DIR / 'predictions.csv', index=False)
with open(RESULTS_DIR / 'feature_sets.json', 'w', encoding='utf-8') as f:
    json.dump({'transfer_features': TRANSFER_FEATURES, 'operational_features': OPERATIONAL_FEATURES}, f, indent=2)

# Importance for one transfer direction; change cities if desired.
importance_model, importance_test, _ = fit_predict(cities['las_vegas'], cities['phoenix'], TRANSFER_FEATURES)
sample = importance_test.sample(min(3000, len(importance_test)), random_state=RANDOM_STATE)
importance = permutation_importance(importance_model, sample[TRANSFER_FEATURES], sample[TARGET], n_repeats=5, random_state=RANDOM_STATE, scoring='neg_mean_absolute_error')
importance_table = pd.DataFrame({'feature': TRANSFER_FEATURES, 'importance_mae': importance.importances_mean, 'importance_std': importance.importances_std}).sort_values('importance_mae', ascending=False)
importance_table.to_csv(RESULTS_DIR / 'transfer_lv_to_phx_permutation_importance.csv', index=False)
display(importance_table.head(20))
display(all_results)
print('Saved outputs to:', RESULTS_DIR)

## Interpretation rules

- Compare the weather-transfer model against a calendar-only baseline before adding optional candidate features.
- Do **not** compare its absolute MW error directly with the operational model: the latter has privileged recent-load information.
- For a new city, weather-only prediction cannot infer total system size. Use city metadata or a short calibration period if absolute MW, rather than weather response, is required.
- Before day-ahead deployment, replace the contemporaneous weather columns with forecast equivalents available at the issuance time.